# Phase 3 - Step 2: the shared RR pipeline (`src/rr_utils.py`)

All six Phase-3 experiments will use the same data loading, tokenisation, windows, class weights,
scorer and submission writer, so only the model / fine-tuning technique differs between them.
This notebook **tests** that shared code. There is no model yet and no GPU is needed.

Checks:
1. load the data (train/dev are lists, test files are dicts); every sentence has a label
2. tokenise: sentence -> bpe32k tokens, and decoding them gives back the sentence
3. windows of whole sentences: every window <= 1536 tokens, every sentence in some window,
   each sentence assigned to the window where it is most central
4. class weights (square-root) for the rare roles
5. scorer sanity checks (perfect prediction = 1.0, ...)
6. two simple baselines on dev (majority label, position only), which the models must beat
7. submission writer + checker, tested on the position baseline (NOT for upload)

In [ ]:
import sys, json, random
from pathlib import Path
import numpy as np
import pandas as pd
from tokenizers import Tokenizer
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
while not (ROOT / ".gitignore").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Project root with .gitignore not found")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, rr_utils
importlib.reload(rr_utils)
from rr_utils import *

OUT = ROOT / "experiments" / "phase3"
OUT.mkdir(parents=True, exist_ok=True)
tokenizer = Tokenizer.from_file(str(ROOT / "data" / "tokenizer" / "bpe32k.json"))
pd.set_option("display.width", 200)
checks = {}          # every number printed below is also saved to p3_02_pipeline_checks.json

## 1. Load the data
`load_rr_data` reads both domains, converts the test dicts into the same format as train/dev,
and checks that every document has exactly one known label per sentence.

In [ ]:
LABELS, DATA = load_rr_data(ROOT)
print(len(LABELS), "labels:", LABELS)
for split, docs in DATA.items():
    for dom in DOMAINS:
        sub = [d for d in docs if d["domain"] == dom]
        print(f"{split:5s} {dom}: {len(sub):3d} documents, {sum(len(d['text']) for d in sub):6,} sentences")
checks["documents"] = {s: len(v) for s, v in DATA.items()}

## 2. Tokenise
Every sentence gets a leading space (`" Leave granted."` -> `ĠLeave`, `Ġgranted`, `.`), exactly like a
sentence in the middle of running text in Phase 1. Sentences longer than 256 tokens are cut
(only ~0.3% of sentences; the start of a sentence says most about its role).
Round-trip check: decoding a sentence's tokens must give back the cleaned sentence.

In [ ]:
MAX_SENT_TOKENS = 256
ENC = {s: encode_rr_docs(DATA[s], tokenizer, LABELS, MAX_SENT_TOKENS, desc=f"tokenising {s}") for s in DATA}

rows, mismatches = [], 0
for s in ENC:
    for d, raw in zip(ENC[s], DATA[s]):
        assert d.n_sent == len(raw["text"]) and d.bounds[-1, 1] == len(d.ids)
        assert (d.bounds[1:, 0] == d.bounds[:-1, 1]).all() and (d.bounds[:, 1] > d.bounds[:, 0]).all()
        for i in range(d.n_sent):
            if d.bounds[i, 1] - d.bounds[i, 0] < MAX_SENT_TOKENS:      # truncated ones can't round-trip
                back = tokenizer.decode(d.ids[d.bounds[i, 0]:d.bounds[i, 1]].tolist()).strip()
                mismatches += back != clean_sentence(raw["text"][i])
    rows.append({"split": s, "tokens": sum(len(d.ids) for d in ENC[s]),
                 "sentences": sum(d.n_sent for d in ENC[s]),
                 "truncated sentences": sum(d.n_truncated for d in ENC[s])})
display(pd.DataFrame(rows).set_index("split"))
print("round-trip mismatches:", mismatches)
checks["tokenise"] = {"rows": rows, "round_trip_mismatches": mismatches}

## 3. Windows of whole sentences
The backbone reads at most 1536 tokens (Phase-1 context). `plan_sentence_windows` fills each window with
as many whole sentences as fit; the next window starts about `stride` tokens later, so neighbouring
windows overlap (here by ~512 tokens). Every sentence then takes its vector from the window where it is
most central (`window_owners`).
Settings compared: 1536/1024 (A, A2, B), 1024/768 (encoder-decoder student: its encoder only ever
read <= 1024 tokens in Phase 1), and "sentence" mode (experiment 6: every sentence alone).

In [ ]:
SETTINGS = {"ctx1536": dict(max_len=1536, stride=1024), "ctx1024": dict(max_len=1024, stride=768),
            "sentence": dict(mode="sentence")}
rows = []
for name, kw in SETTINGS.items():
    for s in ("train", "dev", "test"):
        n_win, win_tok, win_sent, centre = 0, [], [], []
        for d in ENC[s]:
            wins = plan_sentence_windows(d.bounds, **kw)
            owner = window_owners(wins, d.n_sent)
            covered = np.zeros(d.n_sent, bool)
            for w, (s0, s1) in enumerate(wins):
                ids, seg = window_input(d, (s0, s1))
                assert len(ids) <= kw.get("max_len", MAX_SENT_TOKENS) and len(seg) == len(ids)
                assert seg[0] == 0 and seg[-1] == s1 - s0 - 1
                covered[s0:s1] = True
                win_tok.append(len(ids)); win_sent.append(s1 - s0)
            assert covered.all()
            idx = np.arange(d.n_sent)
            lo = np.array([wins[o][0] for o in owner]); hi = np.array([wins[o][1] for o in owner])
            centre.append(np.minimum(idx - lo, hi - 1 - idx))       # sentences of context on the shorter side
            n_win += len(wins)
        c = np.concatenate(centre)
        rows.append({"setting": name, "split": s, "windows": n_win, "mean tokens/window": round(np.mean(win_tok)),
                     "mean sentences/window": round(np.mean(win_sent), 1),
                     "context sentences (min side), median": int(np.median(c))})
win_table = pd.DataFrame(rows).set_index(["setting", "split"])
display(win_table)
checks["windows"] = rows

## 4. Class weights
Square-root weights: weight = (count of most frequent label / count of label) ** 0.5, rescaled so the average
weight per training sentence is 1. Rare roles get a bigger say in the loss without being guessed everywhere.

In [ ]:
W, COUNTS = class_weights(ENC["train"], len(LABELS), power=0.5)
W_full, _ = class_weights(ENC["train"], len(LABELS), power=1.0)
cw = pd.DataFrame({"train sentences": COUNTS, "sqrt weight (used)": W.round(2), "full inverse (not used)": W_full.round(2)},
                  index=LABELS).sort_values("train sentences", ascending=False)
display(cw)
checks["class_weights"] = {l: float(w) for l, w in zip(LABELS, W)}

## 5. Scorer sanity checks
`rr_scores` computes per-label P/R/F1 and macro-F1 over the labels that occur in gold or predictions
(what `sklearn.metrics.f1_score(average="macro")` does). We check it on cases with known answers,
and compare it to sklearn if sklearn is installed.

In [ ]:
dev = DATA["dev"]
gold = {d["id"]: d["labels"] for d in dev}
perfect = score_by_domain(dev, gold, LABELS)["all"]["macro_f1"]
all_fact = score_by_domain(dev, {k: ["Fact"] * len(v) for k, v in gold.items()}, LABELS)["all"]
rng = random.Random(0)
shuffled = {k: rng.sample(v, len(v)) for k, v in gold.items()}
print(f"perfect prediction    : macro-F1 {perfect:.4f} (must be 1.0)")
print(f"everything 'Fact'     : macro-F1 {all_fact['macro_f1']:.4f}, accuracy {all_fact['accuracy']:.4f} "
      f"(= share of Fact on dev; F1 only for Fact)")
print(f"labels shuffled in doc: macro-F1 {score_by_domain(dev, shuffled, LABELS)['all']['macro_f1']:.4f}")
assert abs(perfect - 1) < 1e-9
try:
    from sklearn.metrics import f1_score
    g = [l for d in dev for l in d["labels"]]; p = [l for d in dev for l in shuffled[d["id"]]]
    mine = score_by_domain(dev, shuffled, LABELS)["all"]["macro_f1"]
    print(f"sklearn macro-F1 on the shuffled case: {f1_score(g, p, average='macro'):.4f} (mine {mine:.4f})")
except ImportError:
    print("sklearn not installed - skipped the comparison")
checks["scorer"] = {"perfect": perfect, "all_fact": all_fact["macro_f1"]}

## 6. Baselines on dev (reference points, not submissions)
- **majority**: every sentence gets its domain's most frequent training label
- **position only**: the document is cut into 20 equal parts; each part gets the most frequent training
  label at that position (e.g. the first 5% -> Fact). It never reads a single word.
Any real model must clearly beat both.

In [ ]:
base_path = OUT / "p3_02_baselines.json"
if base_path.exists():
    BASE = json.load(open(base_path))
    print("loaded saved baseline scores")
else:
    BASE = {"majority": score_by_domain(dev, baseline_majority(DATA["train"], dev, LABELS), LABELS),
            "position": score_by_domain(dev, baseline_position(DATA["train"], dev, LABELS), LABELS)}
    json.dump(BASE, open(base_path, "w"), indent=1)
for name, sc in BASE.items():
    print_rr_scores(sc, f"\n{name} baseline (dev):", per_label=(name == "position"))

## 7. Submission writer
`write_submission` writes `prediction.json` = `{doc_id: [label, ...]}` for the 10 test documents (CL + IT
together) and zips it with the file at the ZIP ROOT. `check_submission` re-opens the zip like the grader:
every official test id exactly once, one valid label per sentence.
Tested here with the position baseline -> `experiments/phase3/p3_02_writer_test/` (do NOT upload this one:
it is not a fine-tuned Phase-1 model).

In [ ]:
test_pred = baseline_position(DATA["train"], DATA["test"], LABELS)
zip_path = write_submission(test_pred, DATA["test"], LABELS, OUT / "p3_02_writer_test")
bad = dict(test_pred); first = DATA["test"][0]["id"]; bad[first] = bad[first][:-1]
try:
    write_submission(bad, DATA["test"], LABELS, OUT / "p3_02_writer_test_bad")
    print("ERROR: a short prediction list was accepted")
except AssertionError as e:
    print("a missing sentence is caught:", e)

In [ ]:
checks["baselines_dev_macro_f1"] = {k: {d: v[d]["macro_f1"] for d in v} for k, v in BASE.items()}
json.dump(checks, open(OUT / "p3_02_pipeline_checks.json", "w"), indent=1, default=float)
print("saved", (OUT / "p3_02_pipeline_checks.json").relative_to(ROOT))